In [5]:
import pandas as pd
from pathlib import Path

BASE = Path(__file__).parent if '__file__' in dir() else Path().resolve()

# Cargamos ambos datasets limpios
df_2024 = pd.read_csv(BASE / "../data/processed/2024/datos_limpios.csv", low_memory=False)
df_2025 = pd.read_csv(BASE / "../data/processed/2025/datos_limpios.csv", low_memory=False)

# Agregamos columna de año para distinguirlos
df_2024['anio'] = 2024
df_2025['anio'] = 2025

# Unimos ambos datasets
df = pd.concat([df_2024, df_2025], ignore_index=True)

print(f"Filas 2024: {len(df_2024):,}")
print(f"Filas 2025: {len(df_2025):,}")
print(f"Filas totales: {len(df):,}")
print(f"Columnas: {df.shape[1]}")

KeyboardInterrupt: 

In [ ]:
# Pozos que aparecen en ambos años
pozos_2024 = set(df_2024['idpozo'].unique())
pozos_2025 = set(df_2025['idpozo'].unique())

pozos_ambos = pozos_2024 & pozos_2025
pozos_solo_2024 = pozos_2024 - pozos_2025
pozos_solo_2025 = pozos_2025 - pozos_2024

print("=== CONTINUIDAD DE POZOS ===")
print(f"Pozos activos en ambos años: {len(pozos_ambos):,}")
print(f"Pozos que dejaron de operar (solo en 2024): {len(pozos_solo_2024):,}")
print(f"Pozos nuevos (solo en 2025): {len(pozos_solo_2025):,}")

=== CONTINUIDAD DE POZOS ===
Pozos activos en ambos años: 38,282
Pozos que dejaron de operar (solo en 2024): 1,832
Pozos nuevos (solo en 2025): 982


In [ ]:
# Filtramos solo pozos que aparecen en ambos años
df_continuos = df[df['idpozo'].isin(pozos_ambos)]

# Producción por pozo y año
prod_por_pozo = df_continuos.groupby(['idpozo', 'anio'])[['prod_pet', 'prod_gas']].sum().reset_index()

# Pivoteamos para tener 2024 y 2025 en columnas separadas
prod_pivot = prod_por_pozo.pivot(index='idpozo', columns='anio', values=['prod_pet', 'prod_gas'])
prod_pivot.columns = ['pet_2024', 'pet_2025', 'gas_2024', 'gas_2025']
prod_pivot = prod_pivot.fillna(0).reset_index()

# Calculamos variación porcentual por pozo
prod_pivot['var_pet'] = ((prod_pivot['pet_2025'] - prod_pivot['pet_2024']) / 
                          prod_pivot['pet_2024'].replace(0, float('nan')) * 100).round(2)
prod_pivot['var_gas'] = ((prod_pivot['gas_2025'] - prod_pivot['gas_2024']) / 
                          prod_pivot['gas_2024'].replace(0, float('nan')) * 100).round(2)

print("=== VARIACIÓN DE PRODUCCIÓN EN POZOS CONTINUOS ===")
print(f"Pozos con crecimiento en petróleo: {(prod_pivot['var_pet'] > 0).sum():,}")
print(f"Pozos con caída en petróleo: {(prod_pivot['var_pet'] < 0).sum():,}")
print(f"\nVariación promedio petróleo: {prod_pivot['var_pet'].mean():.2f}%")
print(f"Variación promedio gas: {prod_pivot['var_gas'].mean():.2f}%")

=== VARIACIÓN DE PRODUCCIÓN EN POZOS CONTINUOS ===
Pozos con crecimiento en petróleo: 7,336
Pozos con caída en petróleo: 18,573

Variación promedio petróleo: 321.75%
Variación promedio gas: 2546.77%


In [6]:
# Filtramos pozos con producción mínima para evitar distorsión de outliers
# Solo pozos que produjeron al menos 100 m³ de petróleo en 2024
prod_pivot_filtrado = prod_pivot[prod_pivot['pet_2024'] >= 100].copy()

print("=== VARIACIÓN DE PRODUCCIÓN EN POZOS CONTINUOS (filtrado) ===")
print(f"Pozos analizados (prod. mínima 100 m³ en 2024): {len(prod_pivot_filtrado):,}")
print(f"Pozos con crecimiento en petróleo: {(prod_pivot_filtrado['var_pet'] > 0).sum():,}")
print(f"Pozos con caída en petróleo: {(prod_pivot_filtrado['var_pet'] < 0).sum():,}")
print(f"\nVariación promedio petróleo: {prod_pivot_filtrado['var_pet'].mean():.2f}%")
print(f"Variación promedio gas: {prod_pivot_filtrado['var_gas'].mean():.2f}%")
print(f"\nMediana variación petróleo: {prod_pivot_filtrado['var_pet'].median():.2f}%")
print(f"Mediana variación gas: {prod_pivot_filtrado['var_gas'].median():.2f}%")

=== VARIACIÓN DE PRODUCCIÓN EN POZOS CONTINUOS (filtrado) ===
Pozos analizados (prod. mínima 100 m³ en 2024): 22,107
Pozos con crecimiento en petróleo: 6,077
Pozos con caída en petróleo: 16,028

Variación promedio petróleo: -7.16%
Variación promedio gas: 2888.56%

Mediana variación petróleo: -19.03%
Mediana variación gas: -21.61%


In [7]:
# Filtramos por producción mínima de gas también
prod_pivot_filtrado_gas = prod_pivot[prod_pivot['gas_2024'] >= 100].copy()

print("=== VARIACIÓN GAS (filtrado por producción mínima de gas) ===")
print(f"Pozos analizados: {len(prod_pivot_filtrado_gas):,}")
print(f"Variación promedio gas: {prod_pivot_filtrado_gas['var_gas'].mean():.2f}%")
print(f"Mediana variación gas: {prod_pivot_filtrado_gas['var_gas'].median():.2f}%")

=== VARIACIÓN GAS (filtrado por producción mínima de gas) ===
Pozos analizados: 9,755
Variación promedio gas: 0.31%
Mediana variación gas: -22.76%


In [10]:
# Guardamos el dataset de variación por pozo
ruta_integrated = BASE / "../data/integrated/"
ruta_integrated.mkdir(parents=True, exist_ok=True)

prod_pivot.to_csv(ruta_integrated / "variacion_pozos_2024_2025.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


In [11]:
# Unimos información de provincia al dataset de variación
info_pozos = df_2024[['idpozo', 'provincia', 'tipo_de_recurso', 'formacion']].drop_duplicates('idpozo')
prod_pivot_completo = prod_pivot_filtrado.merge(info_pozos, on='idpozo', how='left')

# Variación promedio por provincia
print("=== VARIACIÓN DE PETRÓLEO POR PROVINCIA ===")
var_provincia = prod_pivot_completo.groupby('provincia').agg(
    cantidad_pozos=('idpozo', 'count'),
    var_pet_promedio=('var_pet', 'mean'),
    var_pet_mediana=('var_pet', 'median')
).round(2).sort_values('var_pet_mediana')
print(var_provincia)

print("\n=== VARIACIÓN POR TIPO DE RECURSO ===")
var_recurso = prod_pivot_completo.groupby('tipo_de_recurso').agg(
    cantidad_pozos=('idpozo', 'count'),
    var_pet_promedio=('var_pet', 'mean'),
    var_pet_mediana=('var_pet', 'median')
).round(2).sort_values('var_pet_mediana')
print(var_recurso)

=== VARIACIÓN DE PETRÓLEO POR PROVINCIA ===
                  cantidad_pozos  var_pet_promedio  var_pet_mediana
provincia                                                          
Neuquén                     4134              0.37           -25.21
La Pampa                     903            -15.66           -20.86
Rio Negro                   1475            -15.19           -19.65
Salta                         50            -23.58           -19.34
Santa Cruz                  6369            -10.87           -19.11
Chubut                      6600             -4.55           -17.17
Mendoza                     2320            -10.20           -14.94
Tierra del Fuego             218            -16.01           -14.59
Formosa                       12            -11.74           -10.03
Jujuy                          8             -5.32            -7.56
Estado Nacional               18            252.36            -5.67

=== VARIACIÓN POR TIPO DE RECURSO ===
                 cantidad_pozos  

In [12]:
# Guardamos variación por provincia y tipo de recurso
var_provincia.reset_index().to_csv(
    ruta_integrated / "variacion_provincia_2024_2025.csv", index=False
)
var_recurso.reset_index().to_csv(
    ruta_integrated / "variacion_recurso_2024_2025.csv", index=False
)
print("Archivos guardados correctamente")

Archivos guardados correctamente


In [13]:
print("=== RESUMEN DE HALLAZGOS COMPARATIVOS 2024 vs 2025 ===")

print("\n--- CONTINUIDAD DE POZOS ---")
print(f"Pozos activos en ambos años: {len(pozos_ambos):,}")
print(f"Pozos que dejaron de operar: {len(pozos_solo_2024):,}")
print(f"Pozos nuevos en 2025: {len(pozos_solo_2025):,}")

print("\n--- VARIACIÓN DE PRODUCCIÓN EN POZOS CONTINUOS ---")
print(f"Pozos con crecimiento en petróleo: {(prod_pivot_filtrado['var_pet'] > 0).sum():,}")
print(f"Pozos con caída en petróleo: {(prod_pivot_filtrado['var_pet'] < 0).sum():,}")
print(f"Mediana variación petróleo: {prod_pivot_filtrado['var_pet'].median():.2f}%")
print(f"Mediana variación gas: {prod_pivot_filtrado_gas['var_gas'].median():.2f}%")

print("\n--- HALLAZGO CLAVE ---")
print("El crecimiento total de petróleo (+2.74%) se explica por")
print("pozos no convencionales de alto rendimiento que compensan")
print("el declive generalizado en pozos convencionales existentes.")
print("La mediana de variación en pozos continuos es -19%, lo que")
print("indica que la mayoría de los pozos perdieron productividad.")

=== RESUMEN DE HALLAZGOS COMPARATIVOS 2024 vs 2025 ===

--- CONTINUIDAD DE POZOS ---
Pozos activos en ambos años: 38,282
Pozos que dejaron de operar: 1,832
Pozos nuevos en 2025: 982

--- VARIACIÓN DE PRODUCCIÓN EN POZOS CONTINUOS ---
Pozos con crecimiento en petróleo: 6,077
Pozos con caída en petróleo: 16,028
Mediana variación petróleo: -19.03%
Mediana variación gas: -22.76%

--- HALLAZGO CLAVE ---
El crecimiento total de petróleo (+2.74%) se explica por
pozos no convencionales de alto rendimiento que compensan
el declive generalizado en pozos convencionales existentes.
La mediana de variación en pozos continuos es -19%, lo que
indica que la mayoría de los pozos perdieron productividad.
